# Garbage Classification Project
This project aims to classify images of garbage into 6 categories: Paper, Glass, Plastic, Metal, Trash, and Cardboard using Deep Learning.

## 1. Datensatz auswählen
We are using the **Garbage Classification** dataset from Kaggle. It contains images of common household waste items categorized into:
- Glass
- Paper
- Cardboard
- Plastic
- Metal
- Trash

## 2. Forschungsfrage oder Hypothese formulieren
**Hypothese:** Mithilfe von Convolutional Neural Networks (CNN) und Transfer Learning können wir verschiedene Abfallkategorien mit einer Genauigkeit von über 80% klassifizieren.
**Task Type:** Multi-class Image Classification (Klassifikation).

## 3. Geeigneten ML-Ansatz wählen und begründen
**ML-Methode:** Transfer Learning mit **MobileNetV2**.
**Begründung:** MobileNetV2 ist ein effizientes und leistungsstarkes Modell für die Bildklassifizierung, das bereits auf ImageNet vortrainiert wurde. Es ist ideal für kleinere Datensätze, da wir das "Wissen" über visuelle Merkmale (Kanten, Formen) übernehmen können.
**Metrik:** **Genauigkeit (Accuracy)** und die **Confusion Matrix**, um zu sehen, welche Klassen am häufigsten verwechselt werden.

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing import image

print("TensorFlow Version:", tf.__version__)


TensorFlow Version: 2.21.0


In [ ]:
TRAIN_NEW_MODEL = False
MODEL_PATH = 'garbage_model.keras'

base_dir = r"datasets/garbage/Garbage classification/Garbage classification"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32


## 4. Lösung implementieren (Pipeline)
### Step 4.1: Datenexploration und Visualisierung

In [ ]:
classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
print(f"Classes found: {classes}")

# Count images per class
class_counts = {}
for c in classes:
    class_dir = os.path.join(base_dir, c)
    class_counts[c] = len(os.listdir(class_dir))

plt.figure(figsize=(10, 5))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title('Number of Images per Class')
plt.xlabel('Garbage Category')
plt.ylabel('Count')
plt.show()


### Step 4.2: Datenvorverarbeitung und Datenaufteilung
Wir nutzen Augmented Data (Rotation, Shift, Zoom, Flip) und teilen den Datensatz in 80% Training und 20% Validierung auf.

In [ ]:
# Data Augmentation / Normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


### Step 4.3: Modelltraining und -optimierung

In [ ]:
if TRAIN_NEW_MODEL or not os.path.exists(MODEL_PATH):
    print('Building and training new model...')
    # Build Model using Transfer Learning (MobileNetV2)
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    predictions = layers.Dense(len(classes), activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    model.summary()
    
    # Train model
    history = model.fit(
        train_generator,
        epochs=10,
        validation_data=val_generator,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3),
            tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True)
        ]
    )
    
    model.save(MODEL_PATH)
    
    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.show()
else:
    print(f"Loading existing model from '{MODEL_PATH}'...")
    model = tf.keras.models.load_model(MODEL_PATH)
    
test_loss, test_acc = model.evaluate(val_generator)
print(f"Validation accuracy: {test_acc:.4f}")


### Step 4.4: Evaluation und Diskussion der Ergebnisse

In [ ]:
# Confusion Matrix
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)

cm = confusion_matrix(val_generator.classes, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Classification Report:\n", classification_report(val_generator.classes, y_pred, target_names=classes))


## 5. Ergebnisse kommunizieren / Single Prediction
Nutze diese Funktion, um einzelne Bilder zu klassifizieren.

In [ ]:
def predict_garbage(img_path):
    """Utility function to predict the category of a single image file."""
    if not os.path.exists(img_path):
        print(f"Error: File '{img_path}' not found.")
        return

    try:
        img = image.load_img(img_path, target_size=IMG_SIZE)
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array /= 255.0

        predictions = model.predict(img_array, verbose=0)
        predicted_class_idx = np.argmax(predictions[0])
        confidence = np.max(predictions[0])
        
        predicted_class_name = classes[predicted_class_idx]
        print(f"Prediction: {predicted_class_name.upper()} ({confidence*100:.2f}% confidence)")
        
        plt.imshow(img)
        plt.title(f"Predicted: {predicted_class_name.upper()}")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Error processing image: {e}")


In [ ]:
# Test Sample
sample_path = os.path.join(base_dir, 'metal', 'metal404.jpg')
predict_garbage(sample_path)
